In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [6]:
DATA_PATH = "../extracted_data/housing_data.csv"
FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [4]:
# ---------------------------------------------------------------------------
# 0. SETUP
# ---------------------------------------------------------------------------

def section(title: str) -> None:
    """Pretty section headers in the console."""
    print("\n" + "=" * 80)
    print(f"  {title}")
    print("=" * 80)


def save_fig(name: str) -> None:
    """Save current figure and close it."""
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  -> saved {path}")

In [7]:
# ---------------------------------------------------------------------------
# 1. LOAD & FIRST LOOK
# ---------------------------------------------------------------------------
section("1. LOAD THE DATA & FIRST LOOK")

df = pd.read_csv(DATA_PATH)

# Rename for clarity (French + spaces -> snake_case English)
df = df.rename(columns={
    "new_m":           "price",          # likely price in 10k MAD
    "chambres":        "bedrooms",
    "salles de bains": "bathrooms",
    "surface":         "surface_m2",
    "ascenseur":       "elevator",
    "floor":           "floor",
    "terrasse":        "terrace",
    "parking":         "parking",
    "Type":            "property_type",
    "City":            "city",
})

# Convert Yes/No to boolean for clean analysis
for col in ["elevator", "terrace", "parking"]:
    df[col] = df[col].map({"Yes": True, "No": False})

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
print("First 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)



  1. LOAD THE DATA & FIRST LOOK
Shape: 4,675 rows x 11 columns

First 5 rows:
   id  price  bedrooms  bathrooms  surface_m2  elevator  floor  terrace  parking property_type        city
0   1  100.0         2          1          98      True      5     True    False   Appartement  Casablanca
1   2  175.0         1          2         173      True      7     True    False        Bureau  Casablanca
2   3  260.0         3          2         150      True      3     True     True   Appartement  Casablanca
3   4  229.0         2          1         130      True      3    False    False   Appartement  Casablanca
4   5  146.0         3          2         163      True      2    False    False   Appartement      Meknès

Data types:
id                 int64
price            float64
bedrooms           int64
bathrooms          int64
surface_m2         int64
elevator            bool
floor              int64
terrace             bool
parking             bool
property_type        str
city            

In [8]:
# ---------------------------------------------------------------------------
# 2. DATA QUALITY CHECK
# ---------------------------------------------------------------------------
section("2. DATA QUALITY")

print(f"Total rows:           {len(df):,}")
print(f"Duplicate rows:       {df.duplicated().sum()}")
print(f"Duplicate IDs:        {df['id'].duplicated().sum()}")
print(f"\nMissing values per column:")
print(df.isnull().sum())

# Sanity ranges
print(f"\nValue ranges:")
print(f"  price:      {df['price'].min():.2f}  ->  {df['price'].max():.2f}")
print(f"  surface_m2: {df['surface_m2'].min()}  ->  {df['surface_m2'].max()}")
print(f"  bedrooms:   {df['bedrooms'].min()}  ->  {df['bedrooms'].max()}")
print(f"  bathrooms:  {df['bathrooms'].min()}  ->  {df['bathrooms'].max()}")
print(f"  floor:      {df['floor'].min()}  ->  {df['floor'].max()}")


  2. DATA QUALITY
Total rows:           4,675
Duplicate rows:       0
Duplicate IDs:        0

Missing values per column:
id               0
price            0
bedrooms         0
bathrooms        0
surface_m2       0
elevator         0
floor            0
terrace          0
parking          0
property_type    0
city             0
dtype: int64

Value ranges:
  price:      67.70  ->  680.00
  surface_m2: 32  ->  720
  bedrooms:   1  ->  10
  bathrooms:  1  ->  4
  floor:      1  ->  7


In [9]:
# ---------------------------------------------------------------------------
# 3. NUMERIC SUMMARY
# ---------------------------------------------------------------------------
section("3. NUMERIC FEATURES — DESCRIPTIVE STATISTICS")

numeric_cols = ["price", "surface_m2", "bedrooms", "bathrooms", "floor"]
print(df[numeric_cols].describe().round(2))

# Skewness & kurtosis tell us about distribution shape
print("\nSkewness (0 = symmetric, >1 = strongly right-skewed):")
print(df[numeric_cols].skew().round(2))


  3. NUMERIC FEATURES — DESCRIPTIVE STATISTICS
         price  surface_m2  bedrooms  bathrooms    floor
count  4675.00     4675.00   4675.00    4675.00  4675.00
mean    187.13      143.72      2.32       1.68     3.20
std     134.85      158.58      1.91       0.84     1.39
min      67.70       32.00      1.00       1.00     1.00
25%     100.00       50.00      1.00       1.00     3.00
50%     146.00       83.00      2.00       1.00     3.00
75%     179.40      150.00      3.00       2.00     4.00
max     680.00      720.00     10.00       4.00     7.00

Skewness (0 = symmetric, >1 = strongly right-skewed):
price         2.18
surface_m2    2.35
bedrooms      2.63
bathrooms     1.07
floor         0.63
dtype: float64


In [10]:
# ---------------------------------------------------------------------------
# 4. CATEGORICAL SUMMARY
# ---------------------------------------------------------------------------
section("4. CATEGORICAL FEATURES — VALUE COUNTS")

for col in ["city", "property_type", "elevator", "terrace", "parking"]:
    print(f"\n{col}:")
    counts = df[col].value_counts()
    pct = (counts / len(df) * 100).round(1)
    print(pd.DataFrame({"count": counts, "pct_%": pct}))


  4. CATEGORICAL FEATURES — VALUE COUNTS

city:
             count  pct_%
city                     
Casablanca    3179   68.0
Marrakech      561   12.0
Meknès         187    4.0
Fès            187    4.0
Mohammadia     187    4.0
Dar Bouazza    187    4.0
Tanger         187    4.0

property_type:
               count  pct_%
property_type              
Appartement     2244   48.0
Studio          1683   36.0
Villa            561   12.0
Bureau           187    4.0

elevator:
          count  pct_%
elevator              
True       4114   88.0
False       561   12.0

terrace:
         count  pct_%
terrace              
False     2618   56.0
True      2057   44.0

parking:
         count  pct_%
parking              
False     2992   64.0
True      1683   36.0


In [11]:
# ---------------------------------------------------------------------------
# 5. UNIVARIATE DISTRIBUTIONS (numeric)
# ---------------------------------------------------------------------------
section("5. NUMERIC DISTRIBUTIONS — HISTOGRAMS & BOXPLOTS")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=axes[0, i], color="steelblue")
    axes[0, i].set_title(f"Distribution: {col}")
    sns.boxplot(x=df[col], ax=axes[1, i], color="lightcoral")
    axes[1, i].set_title(f"Boxplot: {col}")
save_fig("01_numeric_distributions.png")


  5. NUMERIC DISTRIBUTIONS — HISTOGRAMS & BOXPLOTS
  -> saved ../figures/01_numeric_distributions.png


In [12]:
# ---------------------------------------------------------------------------
# 6. CATEGORICAL DISTRIBUTIONS
# ---------------------------------------------------------------------------
section("6. CATEGORICAL DISTRIBUTIONS — BAR CHARTS")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
cat_cols = ["city", "property_type", "elevator", "terrace", "parking"]
for ax, col in zip(axes.flat, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax)
    ax.set_title(f"Counts: {col}")
    ax.tick_params(axis="x", rotation=30)
axes.flat[-1].axis("off")  # hide the unused 6th subplot
save_fig("02_categorical_distributions.png")


  6. CATEGORICAL DISTRIBUTIONS — BAR CHARTS
  -> saved ../figures/02_categorical_distributions.png


In [13]:
# ---------------------------------------------------------------------------
# 7. OUTLIER DETECTION (IQR method)
# ---------------------------------------------------------------------------
section("7. OUTLIER DETECTION (IQR rule: outside Q1-1.5*IQR or Q3+1.5*IQR)")

def iqr_outliers(series: pd.Series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask.sum(), lower, upper

print(f"{'column':<12} {'n_outliers':>10} {'lower':>10} {'upper':>10}")
for col in numeric_cols:
    n, lo, hi = iqr_outliers(df[col])
    print(f"{col:<12} {n:>10} {lo:>10.2f} {hi:>10.2f}")


  7. OUTLIER DETECTION (IQR rule: outside Q1-1.5*IQR or Q3+1.5*IQR)
column       n_outliers      lower      upper
price               748     -19.10     298.50
surface_m2          561    -100.00     300.00
bedrooms            187      -2.00       6.00
bathrooms           187      -0.50       3.50
floor               748       1.50       5.50


In [14]:
# ---------------------------------------------------------------------------
# 8. CORRELATIONS (numeric features)
# ---------------------------------------------------------------------------
section("8. CORRELATIONS BETWEEN NUMERIC FEATURES")

corr = df[numeric_cols].corr()
print(corr.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix — Numeric Features")
save_fig("03_correlation_heatmap.png")


  8. CORRELATIONS BETWEEN NUMERIC FEATURES
            price  surface_m2  bedrooms  bathrooms  floor
price       1.000       0.841     0.929      0.869 -0.111
surface_m2  0.841       1.000     0.806      0.813 -0.003
bedrooms    0.929       0.806     1.000      0.865 -0.039
bathrooms   0.869       0.813     0.865      1.000  0.021
floor      -0.111      -0.003    -0.039      0.021  1.000
  -> saved ../figures/03_correlation_heatmap.png


In [15]:
# ---------------------------------------------------------------------------
# 9. PRICE vs EVERYTHING (the most interesting target)
# ---------------------------------------------------------------------------
section("9. WHAT DRIVES PRICE?")

# 9a. Price by city
print("\nAverage price by city:")
city_stats = df.groupby("city")["price"].agg(["mean", "median", "count"]).round(1)
city_stats = city_stats.sort_values("median", ascending=False)
print(city_stats)

# 9b. Price by property type
print("\nAverage price by property type:")
type_stats = df.groupby("property_type")["price"].agg(["mean", "median", "count"]).round(1)
type_stats = type_stats.sort_values("median", ascending=False)
print(type_stats)

# 9c. Price by amenities
print("\nMedian price by amenity:")
for col in ["elevator", "terrace", "parking"]:
    grp = df.groupby(col)["price"].median().round(1)
    diff = grp[True] - grp[False]
    print(f"  {col:<10} -> with: {grp[True]:.1f}, without: {grp[False]:.1f}  (diff: {diff:+.1f})")

# Visualize all of it
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.boxplot(data=df, x="city", y="price",
            order=city_stats.index, ax=axes[0, 0])
axes[0, 0].set_title("Price by City")
axes[0, 0].tick_params(axis="x", rotation=30)

sns.boxplot(data=df, x="property_type", y="price",
            order=type_stats.index, ax=axes[0, 1])
axes[0, 1].set_title("Price by Property Type")

sns.scatterplot(data=df, x="surface_m2", y="price",
                hue="property_type", alpha=0.5, ax=axes[1, 0])
axes[1, 0].set_title("Price vs Surface (by property type)")

sns.boxplot(data=df.melt(id_vars="price",
                         value_vars=["elevator", "terrace", "parking"],
                         var_name="amenity", value_name="has_it"),
            x="amenity", y="price", hue="has_it", ax=axes[1, 1])
axes[1, 1].set_title("Price by Amenities (Yes vs No)")

save_fig("04_price_drivers.png")


  9. WHAT DRIVES PRICE?

Average price by city:
              mean  median  count
city                             
Mohammadia   680.0   680.0    187
Fès          400.0   400.0    187
Dar Bouazza  179.0   179.0    187
Meknès       146.0   146.0    187
Casablanca   155.6   135.0   3179
Marrakech    178.1   105.6    561
Tanger        94.0    94.0    187

Average price by property type:
                mean  median  count
property_type                      
Villa          473.3   400.0    561
Bureau         175.0   175.0    187
Appartement    173.3   160.5   2244
Studio         111.5   105.6   1683

Median price by amenity:
  elevator   -> with: 134.0, without: 400.0  (diff: -266.0)
  terrace    -> with: 161.0, without: 124.5  (diff: +36.5)
  parking    -> with: 161.0, without: 140.5  (diff: +20.5)
  -> saved ../figures/04_price_drivers.png


In [16]:
# ---------------------------------------------------------------------------
# 10. PRICE PER M² (a real-estate analyst's favorite metric)
# ---------------------------------------------------------------------------
section("10. PRICE PER M² — NORMALIZED COMPARISON")

df["price_per_m2"] = df["price"] / df["surface_m2"]

print("\nPrice per m² by city (sorted by median):")
ppm_city = df.groupby("city")["price_per_m2"].agg(["mean", "median", "std"]).round(3)
ppm_city = ppm_city.sort_values("median", ascending=False)
print(ppm_city)

print("\nPrice per m² by property type:")
ppm_type = df.groupby("property_type")["price_per_m2"].agg(["mean", "median", "std"]).round(3)
ppm_type = ppm_type.sort_values("median", ascending=False)
print(ppm_type)

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="city", y="price_per_m2", order=ppm_city.index)
plt.title("Price per m² by City")
plt.xticks(rotation=30)
save_fig("05_price_per_m2_by_city.png")


  10. PRICE PER M² — NORMALIZED COMPARISON

Price per m² by city (sorted by median):
              mean  median    std
city                             
Marrakech    1.978   2.400  0.598
Casablanca   1.944   2.162  0.719
Tanger       1.446   1.446  0.000
Dar Bouazza  1.388   1.388  0.000
Mohammadia   1.360   1.360  0.000
Meknès       0.896   0.896  0.000
Fès          0.556   0.556  0.000

Price per m² by property type:
                mean  median    std
property_type                      
Studio         2.592   2.700  0.223
Appartement    1.437   1.344  0.472
Villa          1.016   1.133  0.339
Bureau         1.012   1.012  0.000
  -> saved ../figures/05_price_per_m2_by_city.png


In [17]:
# ---------------------------------------------------------------------------
# 11. CROSS-TABS — CITY x PROPERTY TYPE
# ---------------------------------------------------------------------------
section("11. CROSS-TABULATIONS")

print("\nListing counts: City x Property Type")
ct_counts = pd.crosstab(df["city"], df["property_type"])
print(ct_counts)

print("\nMedian price: City x Property Type")
ct_price = df.pivot_table(values="price", index="city",
                          columns="property_type", aggfunc="median").round(1)
print(ct_price)

plt.figure(figsize=(10, 6))
sns.heatmap(ct_price, annot=True, fmt=".0f", cmap="YlOrRd")
plt.title("Median Price — City x Property Type")
save_fig("06_heatmap_city_type.png")


  11. CROSS-TABULATIONS

Listing counts: City x Property Type
property_type  Appartement  Bureau  Studio  Villa
city                                             
Casablanca            1683     187    1309      0
Dar Bouazza            187       0       0      0
Fès                      0       0       0    187
Marrakech                0       0     374    187
Meknès                 187       0       0      0
Mohammadia               0       0       0    187
Tanger                 187       0       0      0

Median price: City x Property Type
property_type  Appartement  Bureau  Studio  Villa
city                                             
Casablanca           161.0   175.0   116.7    NaN
Dar Bouazza          179.0     NaN     NaN    NaN
Fès                    NaN     NaN     NaN  400.0
Marrakech              NaN     NaN    97.2  340.0
Meknès               146.0     NaN     NaN    NaN
Mohammadia             NaN     NaN     NaN  680.0
Tanger                94.0     NaN     NaN    NaN
 

In [18]:
# ---------------------------------------------------------------------------
# 12. PAIRWISE RELATIONSHIPS (sample for performance)
# ---------------------------------------------------------------------------
section("12. PAIRPLOT — RELATIONSHIPS BETWEEN KEY VARIABLES")

sample = df.sample(min(1000, len(df)), random_state=42)
sns.pairplot(sample[["price", "surface_m2", "bedrooms", "bathrooms", "property_type"]],
             hue="property_type", diag_kind="kde", plot_kws={"alpha": 0.5})
plt.suptitle("Pairwise Relationships (sample of 1000)", y=1.01)
save_fig("07_pairplot.png")



  12. PAIRPLOT — RELATIONSHIPS BETWEEN KEY VARIABLES
  -> saved ../figures/07_pairplot.png


In [19]:
# ---------------------------------------------------------------------------
# 13. KEY FINDINGS — AUTO-GENERATED SUMMARY
# ---------------------------------------------------------------------------
section("13. KEY FINDINGS")

most_expensive_city  = ppm_city.index[0]
cheapest_city        = ppm_city.index[-1]
most_common_type     = df["property_type"].value_counts().idxmax()
most_common_city     = df["city"].value_counts().idxmax()
corr_price_surface   = df["price"].corr(df["surface_m2"])

print(f"""
  - Dataset has {len(df):,} listings across {df['city'].nunique()} cities
    and {df['property_type'].nunique()} property types.

  - Most listed city:        {most_common_city} ({(df['city'] == most_common_city).mean() * 100:.1f}% of all listings)
  - Most listed type:        {most_common_type} ({(df['property_type'] == most_common_type).mean() * 100:.1f}% of all listings)

  - Most expensive city per m²:  {most_expensive_city}  ({ppm_city.loc[most_expensive_city, 'median']:.2f})
  - Cheapest city per m²:        {cheapest_city}  ({ppm_city.loc[cheapest_city, 'median']:.2f})

  - Correlation between price and surface: {corr_price_surface:.3f}
    (positive and strong = bigger places cost more, as expected)

  - Median price overall:    {df['price'].median():.1f}
  - Median surface overall:  {df['surface_m2'].median()} m²
  - Median price per m²:     {df['price_per_m2'].median():.3f}

  Amenity premiums (median price difference):
    + elevator:  {df.groupby('elevator')['price'].median().diff().iloc[-1]:+.1f}
    + terrace:   {df.groupby('terrace')['price'].median().diff().iloc[-1]:+.1f}
    + parking:   {df.groupby('parking')['price'].median().diff().iloc[-1]:+.1f}
""")

print(f"\nAll {len(os.listdir(FIG_DIR))} charts saved in: {os.path.abspath(FIG_DIR)}")
print("Done.")


  13. KEY FINDINGS

  - Dataset has 4,675 listings across 7 cities
    and 4 property types.

  - Most listed city:        Casablanca (68.0% of all listings)
  - Most listed type:        Appartement (48.0% of all listings)

  - Most expensive city per m²:  Marrakech  (2.40)
  - Cheapest city per m²:        Fès  (0.56)

  - Correlation between price and surface: 0.841
    (positive and strong = bigger places cost more, as expected)

  - Median price overall:    146.0
  - Median surface overall:  83.0 m²
  - Median price per m²:     1.733

  Amenity premiums (median price difference):
    + elevator:  -266.0
    + terrace:   +36.5
    + parking:   +20.5


All 7 charts saved in: /home/ahmed/Desktop/Morocco_House_Price_Prediction/Morocco_House_Price_Pred/figures
Done.
